In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import CenteredNorm

# =====================================================================
# 1. SETUP, DATA LOADING & STYLING
# =====================================================================
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']

plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12
})

cohort_colors = {
    'Moffitt': '#0072B2',       # Blue
    'VenEx Trial': '#D55E00'    # Vermilion / Red-Orange
}

# --- File Paths ---
all_errors_file = "IntermediateResults/venex_allerrors.pkl"
moffitt_params_file = "IntermediateResults/moffitt_weighted_params.csv"
venex_params_file = "IntermediateResults/venex_weighted_params.csv"

# --- Data Loading ---
df_all = pd.read_pickle(all_errors_file)
df_all = pd.DataFrame(df_all)
df_all['ID'] = df_all['ID'].astype(str).str.strip()

moffit_params = pd.read_csv(moffitt_params_file)
venex_params = pd.read_csv(venex_params_file)
moffit_params['Cohort'] = "Moffitt"
venex_params['Cohort'] = "VenEx Trial"
patient_weighted_fit = pd.concat([venex_params, moffit_params], ignore_index=True)

# =====================================================================
# 2. FIGURE 2x3 GRID 
# =====================================================================
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(22, 13))
fig.subplots_adjust(hspace=0.32, wspace=0.35)

# Top row: Parameter Landscapes
ax_total, ax_neutro, ax_blast = axes[0, 0], axes[0, 1], axes[0, 2]

# Bottom row: Cohort Parameter Distributions
ax_err_hist, ax_qL_hist, ax_coef_hist = axes[1, 0], axes[1, 1], axes[1, 2]

# =====================================================================
# 3. TOP ROW: PARAMETER LANDSCAPES
# =====================================================================
target_patient_id = "9109"
df_grid = df_all[df_all['ID'] == target_patient_id].copy()

if df_grid.empty:
    sample_ids = list(df_all['ID'].unique()[:5])
    raise ValueError(
        f"Patient ID '{target_patient_id}' not found in df_all. "
        f"Available sample IDs in dataset: {sample_ids}"
    )

pivot_neutro = df_grid.pivot_table(index="coef", columns="pHSC", values="err_neutro", aggfunc="mean").sort_index().sort_index(axis=1)
pivot_blast  = df_grid.pivot_table(index="coef", columns="pHSC", values="err_blast", aggfunc="mean").sort_index().sort_index(axis=1)
pivot_total  = df_grid.pivot_table(index="coef", columns="pHSC", values="total_err", aggfunc="mean").sort_index().sort_index(axis=1)

X, Y = np.meshgrid(pivot_neutro.columns.values, pivot_neutro.index.values)

# Panel 1: Total Optimization Fit Error 
cB = ax_total.pcolormesh(X, Y, pivot_total.values, cmap="magma", shading="auto")
cbarB = fig.colorbar(cB, ax=ax_total, pad=0.03, shrink=0.9)
cbarB.set_label("Total Optimization Fit Error", fontweight="bold", labelpad=10)

min_idx = df_grid["total_err"].idxmin()
best_pHSC = df_grid.loc[min_idx, "pHSC"]
best_coef = df_grid.loc[min_idx, "coef"]
ax_total.plot(best_pHSC, best_coef, marker="*", color="gold", markeredgecolor="black", markersize=18, zorder=5)

# Panel 2: Neutrophil Error Residual
cC = ax_neutro.pcolormesh(X, Y, pivot_neutro.values, cmap="coolwarm", shading="auto", norm=CenteredNorm(0))
cbarC = fig.colorbar(cC, ax=ax_neutro, pad=0.03, shrink=0.9)
cbarC.set_label("Neutrophil Error Residual", fontweight="bold", labelpad=10)

# Panel 3: Blast Error Residual
cD = ax_blast.pcolormesh(X, Y, pivot_blast.values, cmap="coolwarm", shading="auto", norm=CenteredNorm(0))
cbarD = fig.colorbar(cD, ax=ax_blast, pad=0.03, shrink=0.9)
cbarD.set_label("Blast Error Residual", fontweight="bold", labelpad=10)

for ax in [ax_total, ax_neutro, ax_blast]:
    ax.set_xlabel(r"HSC Dislodgement Probability ($q_L$)", fontweight="bold", labelpad=8)
    ax.set_ylabel(r"Proliferation Coefficient ($p_1^l / p_1^c$)", fontweight="bold", labelpad=8)
    ax.grid(False)

# =====================================================================
# 4. BOTTOM ROW: COHORT PARAMETER DISTRIBUTIONS
# =====================================================================
sns.histplot(data=patient_weighted_fit, x='err_weighted', hue='Cohort', kde=True, 
             ax=ax_err_hist, edgecolor='black', bins=12, palette=cohort_colors, stat="density")
ax_err_hist.set_xlabel('Weighted Error Residual', fontweight='bold', labelpad=8)
ax_err_hist.set_ylabel('Density', fontweight='bold', labelpad=8)

sns.histplot(data=patient_weighted_fit, x='pHSC_weighted', hue='Cohort', kde=True, 
             ax=ax_qL_hist, edgecolor='black', bins=12, palette=cohort_colors, stat="density")
ax_qL_hist.set_xlabel(r'HSC Dislodgement Probability ($q_L$)', fontweight='bold', labelpad=8)
ax_qL_hist.set_ylabel('Density', fontweight='bold', labelpad=8) 

sns.histplot(data=patient_weighted_fit, x='coef_weighted', hue='Cohort', kde=True, 
             ax=ax_coef_hist, edgecolor='black', bins=12, palette=cohort_colors, stat="density")
ax_coef_hist.set_xlabel(r'Proliferation Coefficient ($p_1^l / p_1^c$)', fontweight='bold', labelpad=8)
ax_coef_hist.set_ylabel('Density', fontweight='bold', labelpad=8)

for ax in [ax_err_hist, ax_qL_hist, ax_coef_hist]:
    if ax.get_legend():
        sns.move_legend(ax, "upper right", title='Cohort')
    ax.grid(True, alpha=0.2, linestyle='--')

# =====================================================================
# 5. CORNER STAMPING (A through F)
# =====================================================================
ordered_axes = [ax_total, ax_neutro, ax_blast, ax_err_hist, ax_qL_hist, ax_coef_hist]
panel_tags = ['A', 'B', 'C', 'D', 'E', 'F']

for ax, tag in zip(ordered_axes, panel_tags):
    ax.text(-0.08, 1.05, tag, transform=ax.transAxes, 
            fontsize=22, fontweight='bold', va='bottom', ha='right')

# =====================================================================
# 6. SAVE & SHOW
# =====================================================================
os.makedirs("Figures", exist_ok=True)
plt.savefig('Figures/Supp_Figure1.png', dpi=300, bbox_inches='tight')
plt.savefig('Figures/Supp_Figure1.pdf', format='pdf', dpi=300, bbox_inches='tight')

plt.show()